In [3]:
!pip install langchain langchain-core --quiet

In [5]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate

In [6]:
explain_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms for beginners"
)
def explain_topic(topic: str) -> str:
    return explain_template.format(topic=topic)
print(explain_topic("Machine Learning"))
print(explain_topic("Blockchain"))

Explain Machine Learning in simple terms for beginners
Explain Blockchain in simple terms for beginners


In [7]:
multi_input_template = PromptTemplate(
    input_variables=["topic", "audience", "tone"],
    template="Explain {topic} for {audience} in a {tone} tone"
)
test_cases = [
    {"topic": "AI",            "audience": "beginners", "tone": "friendly"},
    {"topic": "Python",        "audience": "kids",      "tone": "fun"},
    {"topic": "Deep Learning", "audience": "engineers", "tone": "technical"},
]
for case in test_cases:
    print(multi_input_template.format(**case))

Explain AI for beginners in a friendly tone
Explain Python for kids in a fun tone
Explain Deep Learning for engineers in a technical tone


In [8]:
variation_templates = {
    "Teaching":     PromptTemplate(input_variables=["topic"], template="Explain {topic} clearly step by step"),
    "Interview":    PromptTemplate(input_variables=["topic"], template="Ask 3 interview questions about {topic}"),
    "Storytelling": PromptTemplate(input_variables=["topic"], template="Explain {topic} as a story"),
}
topic = "Machine Learning"
for style, template in variation_templates.items():
    print(f"[{style}] {template.format(topic=topic)}")

[Teaching] Explain Machine Learning clearly step by step
[Interview] Ask 3 interview questions about Machine Learning
[Storytelling] Explain Machine Learning as a story


In [9]:
ROLE_BEHAVIOURS = {
    "teacher":     "You are an expert teacher. Explain concepts clearly, using examples.",
    "interviewer": "You are a technical interviewer. Ask precise, challenging questions.",
    "motivator":   "You are an enthusiastic motivator. Make the topic exciting and inspiring.",
}
chat_template = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template("{behaviour}"),
    HumanMessagePromptTemplate.from_template("Please help me understand {topic}."),
])
def build_chat_prompt(role: str, topic: str):
    if role not in ROLE_BEHAVIOURS:
        raise ValueError(f"Unknown role '{role}'. Choose from: {list(ROLE_BEHAVIOURS.keys())}")
    return chat_template.format_messages(behaviour=ROLE_BEHAVIOURS[role], topic=topic)
for r in ROLE_BEHAVIOURS:
    msgs = build_chat_prompt(r, "Neural Networks")
    print(f"Role: {r}")
    for m in msgs:
        print(f"  [{m.__class__.__name__}] {m.content}")
    print()

Role: teacher
  [SystemMessage] You are an expert teacher. Explain concepts clearly, using examples.
  [HumanMessage] Please help me understand Neural Networks.

Role: interviewer
  [SystemMessage] You are a technical interviewer. Ask precise, challenging questions.
  [HumanMessage] Please help me understand Neural Networks.

Role: motivator
  [SystemMessage] You are an enthusiastic motivator. Make the topic exciting and inspiring.
  [HumanMessage] Please help me understand Neural Networks.



In [10]:
VALID_AUDIENCES = ["beginner", "intermediate", "expert"]
VALID_TONES     = ["formal", "casual", "fun"]
DEFAULT_AUDIENCE = "beginner"
DEFAULT_TONE     = "casual"
def validate_inputs(audience: str, tone: str) -> tuple:
    if audience not in VALID_AUDIENCES:
        print(f"[WARNING] Invalid audience '{audience}'. Defaulting to '{DEFAULT_AUDIENCE}'.")
        audience = DEFAULT_AUDIENCE
    if tone not in VALID_TONES:
        print(f"[WARNING] Invalid tone '{tone}'. Defaulting to '{DEFAULT_TONE}'.")
        tone = DEFAULT_TONE
    return audience, tone
print(validate_inputs("beginner", "fun"))
print(validate_inputs("newbie", "angry"))

('beginner', 'fun')
[WARNING] Invalid audience 'newbie'. Defaulting to 'beginner'.
[WARNING] Invalid tone 'angry'. Defaulting to 'casual'.
('beginner', 'casual')


In [11]:
STYLE_TEMPLATES = {
    "teaching": PromptTemplate(
        input_variables=["topic", "audience", "tone"],
        template="Explain {topic} for {audience} in a {tone} teaching style, step by step"
    ),
    "interview": PromptTemplate(
        input_variables=["topic", "audience", "tone"],
        template="Ask 3 interview questions about {topic} for {audience} in a {tone} tone"
    ),
    "storytelling": PromptTemplate(
        input_variables=["topic", "audience", "tone"],
        template="Explain {topic} as a {tone} story for {audience}"
    ),
}
def generate_prompt(topic: str, audience: str, tone: str, style: str) -> str:
    audience, tone = validate_inputs(audience, tone)
    if style not in STYLE_TEMPLATES:
        raise ValueError(f"Unknown style '{style}'. Choose from: {list(STYLE_TEMPLATES.keys())}")
    return STYLE_TEMPLATES[style].format(topic=topic, audience=audience, tone=tone)
test_inputs = [
    ("Python",          "beginner",     "casual",    "teaching"),
    ("Data Structures", "intermediate", "formal",    "interview"),
    ("Cloud Computing", "expert",       "fun",       "storytelling"),
    ("Recursion",       "kids",         "technical", "teaching"),
]
for topic, audience, tone, style in test_inputs:
    print(f"[{style.upper()}] {generate_prompt(topic, audience, tone, style)}")

[TEACHING] Explain Python for beginner in a casual teaching style, step by step
[INTERVIEW] Ask 3 interview questions about Data Structures for intermediate in a formal tone
[STORYTELLING] Explain Cloud Computing as a fun story for expert
[WARNING] Invalid audience 'kids'. Defaulting to 'beginner'.
[WARNING] Invalid tone 'technical'. Defaulting to 'casual'.
[TEACHING] Explain Recursion for beginner in a casual teaching style, step by step


In [12]:
reusable_template = PromptTemplate(
    input_variables=["topic", "audience", "tone"],
    template="Explain {topic} to {audience} in a {tone} tone"
)
five_inputs = [
    {"topic": "Neural Networks",   "audience": "beginners",   "tone": "friendly"},
    {"topic": "Quantum Computing", "audience": "experts",     "tone": "technical"},
    {"topic": "Climate Change",    "audience": "school kids", "tone": "fun"},
    {"topic": "Stock Markets",     "audience": "investors",   "tone": "formal"},
    {"topic": "Photosynthesis",    "audience": "gardeners",   "tone": "casual"},
]
for i, inputs in enumerate(five_inputs, start=1):
    print(f"{i}. {reusable_template.format(**inputs)}")

1. Explain Neural Networks to beginners in a friendly tone
2. Explain Quantum Computing to experts in a technical tone
3. Explain Climate Change to school kids in a fun tone
4. Explain Stock Markets to investors in a formal tone
5. Explain Photosynthesis to gardeners in a casual tone


In [13]:
def full_pipeline(topic: str, audience: str, tone: str, style: str) -> str:
    audience, tone = validate_inputs(audience, tone)
    return generate_prompt(topic, audience, tone, style)
print(full_pipeline("Transformers", "intermediate", "casual", "teaching"))

Explain Transformers for intermediate in a casual teaching style, step by step
